# Day 8 — pandas habits to unlearn: a cheat sheet

*100 Days of Polars · Phase 1: Foundations and mental model*

- **Post:** on Medium (link added after publishing)
- **Dataset:** [Olist Brazilian e-commerce](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) — CSVs in `data/raw/`
- **Written for:** Polars 2.x (run from the repo root)

In [ ]:
import polars as pl

print("polars", pl.__version__)

Twelve pandas habits, each next to its Polars replacement, on the same Olist tables.
Run the pandas cell and the Polars cell of each pair and compare.

In [ ]:
import pandas as pd

items_pl = pl.read_csv("data/raw/olist_order_items_dataset.csv", try_parse_dates=True)
orders_pl = pl.read_csv("data/raw/olist_orders_dataset.csv", try_parse_dates=True)
items_pd = items_pl.to_pandas()
orders_pd = orders_pl.to_pandas()

## 1. Selecting columns: `df[[...]]` → `select`

In [ ]:
# pandas
items_pd[["order_id", "price"]].head(3)

In [ ]:
# polars
items_pl.select("order_id", "price").head(3)

## 2. Boolean masks → `filter`

In [ ]:
# pandas
items_pd[(items_pd["price"] > 100) & (items_pd["freight_value"] < 20)].shape

In [ ]:
# polars
items_pl.filter(pl.col("price") > 100, pl.col("freight_value") < 20).shape

## 3. Assigning columns (`df["x"] = ...`, `assign`) → `with_columns`

In [ ]:
# pandas
items_pd2 = items_pd.copy()
items_pd2["total"] = items_pd2["price"] + items_pd2["freight_value"]
items_pd2[["price", "freight_value", "total"]].head(3)

In [ ]:
# polars
items_pl.with_columns(total=pl.col("price") + pl.col("freight_value")).select("price", "freight_value", "total").head(3)

## 4. Row-wise `apply` → rewrite as an expression

`apply(f, axis=1)` calls a Python function once per row. Polars' literal equivalent is `pl.struct(...).map_elements(f)`, which is just as slow. The real replacement is the expression that does what `f` does: for an if/else that is `when/then/otherwise`, Polars' version of `np.where`.

In [ ]:
# pandas
items_pd.apply(lambda r: "high" if r["price"] > 100 else "low", axis=1).value_counts()

In [ ]:
# polars, literal translation (slow: Python per row)
items_pl.select(
    band=pl.struct("price").map_elements(lambda r: "high" if r["price"] > 100 else "low", return_dtype=pl.String)
)["band"].value_counts()

In [ ]:
# polars, the real replacement: a vectorized expression (like np.where in pandas)
items_pl.select(band=pl.when(pl.col("price") > 100).then(pl.lit("high")).otherwise(pl.lit("low")))["band"].value_counts()

## 5. `groupby().agg()` → `group_by().agg()` with named outputs

In [ ]:
# pandas
items_pd.groupby("seller_id").agg(revenue=("price", "sum"), items=("order_id", "size")).sort_values("revenue", ascending=False).head(3)

In [ ]:
# polars
items_pl.group_by("seller_id").agg(revenue=pl.col("price").sum(), items=pl.len()).sort("revenue", descending=True).head(3)

## 6. `merge` → `join`

In [ ]:
# pandas
items_pd.merge(orders_pd[["order_id", "order_status"]], on="order_id", how="left").shape

In [ ]:
# polars
items_pl.join(orders_pl.select("order_id", "order_status"), on="order_id", how="left").shape

## 7. `sort_values` → `sort`

In [ ]:
# pandas
items_pd.sort_values(["seller_id", "price"], ascending=[True, False]).head(3)[["seller_id", "price"]]

In [ ]:
# polars
items_pl.sort("seller_id", "price", descending=[False, True]).select("seller_id", "price").head(3)

## 8. `fillna` → `fill_null`

In [ ]:
# pandas
orders_pd["order_approved_at"].fillna(orders_pd["order_purchase_timestamp"]).isna().sum()

In [ ]:
# polars
orders_pl.select(pl.col("order_approved_at").fill_null(pl.col("order_purchase_timestamp")).null_count()).item()

## 9. `astype` → `cast`

In [ ]:
# pandas
items_pd["order_item_id"].astype("int8").dtype

In [ ]:
# polars
items_pl.select(pl.col("order_item_id").cast(pl.Int8)).dtypes

## 10. `value_counts` → `value_counts` (returns a DataFrame)

In [ ]:
# pandas
orders_pd["order_status"].value_counts().head(3)

In [ ]:
# polars
orders_pl["order_status"].value_counts(sort=True).head(3)

## 11. `drop_duplicates` → `unique`

In [ ]:
# pandas
items_pd.drop_duplicates(subset=["order_id"]).shape

In [ ]:
# polars
items_pl.unique(subset="order_id").shape

## 12. `rename(columns=...)` → `rename`

In [ ]:
# pandas
items_pd.rename(columns={"price": "item_price"}).columns[:6].tolist()

In [ ]:
# polars
items_pl.rename({"price": "item_price"}).columns[:6]

## Two more habits: `pl.col` vs `df["col"]`, and `select` vs `with_columns`

- `pl.col("x")` is an **instruction** that runs inside a context, works lazily and refers to whatever table reaches that step of a chain. `df["x"]` is the **data itself**, taken out of `df` right now (eager only). Use `pl.col` to describe work, `df["x"]` to take results out.
- `select` returns **only** the listed columns; `with_columns` returns **all** columns plus the new ones. Reassigning after `select` drops everything you did not list.

In [ ]:
band = pl.when(pl.col("price") > 100).then(pl.lit("high")).otherwise(pl.lit("low"))

print(items_pl.select(band=band).columns)              # only 'band'
print(items_pl.with_columns(band=band).columns)        # everything + 'band'

# pl.col refers to a column created earlier in the same chain
items_pl.with_columns(band=band).filter(pl.col("band") == "high").height

## How much does `apply` cost?

In [ ]:
import time
import numpy as np


def timed(fn):
    t0 = time.perf_counter()
    fn()
    return (time.perf_counter() - t0) * 1000


timings = {
    "pandas apply (row-wise)": timed(
        lambda: items_pd.apply(lambda r: "high" if r["price"] > 100 else "low", axis=1)
    ),
    "pandas np.where (vectorized)": timed(lambda: np.where(items_pd["price"] > 100, "high", "low")),
    "polars map_elements (row-wise)": timed(
        lambda: items_pl.select(
            pl.struct("price").map_elements(lambda r: "high" if r["price"] > 100 else "low", return_dtype=pl.String)
        )
    ),
    "polars when/then (vectorized)": timed(
        lambda: items_pl.select(pl.when(pl.col("price") > 100).then(pl.lit("high")).otherwise(pl.lit("low")))
    ),
}
for name, ms in timings.items():
    print(f"{name:<32} {ms:8.1f} ms")

## Habits with no Polars equivalent (on purpose)

| pandas habit | In Polars |
|---|---|
| `inplace=True` | Not there. Every method returns a new frame; reassign it. Copies are cheap because data is shared. |
| `set_index` / `reset_index` / `.loc[label]` | No index. Keep keys as columns; use `filter` or `join`. |
| `df.iloc[5:10]` | `df.slice(5, 5)` or `df[5:10]` |
| Chained assignment `df[df.x > 1]["y"] = 0` | `df.with_columns(pl.when(pl.col("x") > 1).then(0).otherwise(pl.col("y")).alias("y"))` |
| `df.melt(...)` | `df.unpivot(...)` |
| `df.pivot_table(...)` | `df.pivot(on=..., index=..., values=..., aggregate_function=...)` |
| `object` columns of mixed types | Not allowed: one dtype per column |

## Try it yourself

1. Add a pandas vectorized version of habit 4 with `np.where` and time it too. How close does it get to Polars?
2. Pick one pandas script you wrote at work and translate three lines using this sheet.